In [ ]:
!git clone https://github.com/Otza02/satellite-img-segmentation.git
%cd satellite-img-segmentation
%pip install -e .

## Reiniciar sesion

In [ ]:
%cd /content/satellite-img-segmentation
!unzip -q data/train.zip -d data/train

## Entrenamiento de Attention U-Net

In [ ]:
from satelliteSegmentation.config import Config
from satelliteSegmentation.dataset import SatelliteData, AugmentedSubset, spatial_train_val_split
from satelliteSegmentation.models import AttentionUNet
from satelliteSegmentation.train import train_model

import torch
from torch.utils.data import DataLoader
from torchvision.transforms import v2

import pandas as pd
import numpy as np
from pathlib import Path
import json
import shutil

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 64

## Preparar Datos

In [ ]:
data = SatelliteData("data/train")

train_tf = v2.Compose([
    v2.RandomHorizontalFlip(),
    v2.RandomVerticalFlip(),
    v2.RandomRotation([0, 180], fill=6),
])
color_tf = v2.Compose([
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.1)
])

train_idx, val_idx = spatial_train_val_split(data, val_fraction=0.2)

train_loader = DataLoader(
    AugmentedSubset(data, train_idx, train_tf, color_tf),
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=device == "cuda",
    persistent_workers=device == "cuda",
)

val_loader = DataLoader(
    AugmentedSubset(data, val_idx),
    batch_size=batch_size,
    num_workers=2,
    pin_memory=device == "cuda",
    persistent_workers=device == "cuda",
)

In [ ]:
def calc_weights(
    dataloader,
    num_classes: int,
    ignore_index: int,
):
    counts = torch.zeros(num_classes, dtype=torch.long)

    for _, masks in dataloader:
        masks = masks.flatten()
        masks = masks[masks != ignore_index]

        counts += torch.bincount(
            masks,
            minlength=num_classes,
        )

    # Solo clases que participan en el entrenamiento
    valid_counts = counts.clone()
    valid_counts[ignore_index] = 0

    # Ejemplo: inverse square root frequency
    weights = torch.zeros(num_classes, dtype=torch.float32)

    valid = valid_counts > 0
    weights[valid] = 1.0 / torch.sqrt(valid_counts[valid].float())

    # Normalizar para que la media de los weights válidos sea 1
    weights[valid] /= weights[valid].mean()

    # El weight de la clase ignorada es irrelevante
    weights[ignore_index] = 0.0

    return counts, weights

counts, weights = calc_weights(
    train_loader,
    num_classes=7,
    ignore_index=6,
)
weights = weights.to(device)

In [ ]:
weights

In [ ]:
def save_model(name: str, model: torch.nn.Module, hist: dict, config: Config):
    result_folder = Path(f"result/{name}")
    result_folder.mkdir(parents=True, exist_ok=True)

    torch.save(model.state_dict(), result_folder / "checkpoint.pt")
    pd.DataFrame(hist).to_csv(result_folder / "hist.csv", index=False)
    with open(result_folder / "config.json", "w") as f:
        json.dump(config.to_json(), f)

    return result_folder

def zip_files(folder: Path):
    folder = Path(folder)

    zip_path = shutil.make_archive(
        base_name=str(folder),
        format="zip",
        root_dir=folder.parent,
        base_dir=folder.name,
    )

    print(f"Se guardó el zip en: {zip_path}")
    # files.download(zip_path)

## Train

In [ ]:
configs = {
    f"lr-{a:.2e}": Config(device, lr=a, weights=weights)
    for a in np.linspace(1e-5, 1e-3, 8)
}

In [ ]:
for name, conf in configs.items():
    torch.manual_seed(2026)
    model = AttentionUNet(conf)
    print(f"Modelo: {name} | n_params: {sum([p.numel() for p in model.parameters()]):,}")

    criterion = torch.nn.CrossEntropyLoss(conf.weights, ignore_index=6)

    model, hist = train_model(model, train_loader, val_loader, criterion, conf)

    result_folder = save_model(name, model, hist, conf)
    zip_files(result_folder)